In [1]:
import json
from collections import Counter

# ==========================
# Config & Load Data
# ==========================
FILE_PATH = r"P:\Literature-Research-Agent-main\repository\filtered.json"

try:
    with open(FILE_PATH, "r", encoding="utf-8") as f:
        papers = json.load(f)
    print(f"Successfully loaded dataset! Total Filtered Papers: {len(papers)}\n" + "=" * 55)
except FileNotFoundError:
    print(f"Error: File not found at {FILE_PATH}.")
    exit()

if not papers:
    print("The dataset is empty.")
    exit()

# ==========================
# Keys / Fields Structure Inspection
# ==========================
all_keys = Counter()
for paper in papers:
    if isinstance(paper, dict):
        all_keys.update(paper.keys())

print("Dataset Key/Field Presence Summary:")
print(f"{'Field Name':<25} | {'Count':<10} | {'Coverage (%)':<10}")
print("-" * 55)

for key, count in all_keys.most_common():
    pct = (count / len(papers)) * 100
    print(f"{key:<25} | {count:<10} | {pct:.2f}%")

print("=" * 55)
print(f"Total Unique Keys Found: {len(all_keys)}")
print("All Keys List:", list(all_keys.keys()))

Successfully loaded dataset! Total Filtered Papers: 64500
Dataset Key/Field Presence Summary:
Field Name                | Count      | Coverage (%)
-------------------------------------------------------
id                        | 64500      | 100.00%
doi                       | 64500      | 100.00%
title                     | 64500      | 100.00%
display_name              | 64500      | 100.00%
publication_year          | 64500      | 100.00%
publication_date          | 64500      | 100.00%
ids                       | 64500      | 100.00%
language                  | 64500      | 100.00%
primary_location          | 64500      | 100.00%
type                      | 64500      | 100.00%
indexed_in                | 64500      | 100.00%
open_access               | 64500      | 100.00%
authorships               | 64500      | 100.00%
institutions              | 64500      | 100.00%
countries_distinct_count  | 64500      | 100.00%
institutions_distinct_count | 64500      | 100.00%
correspond

In [2]:
import json
from tqdm import tqdm

# ==========================
# 1. Cleaning Helper Functions
# ==========================
KEYS_TO_DROP = {
    "abstract_inverted_index", "is_paratext", "is_xpac", "is_retracted",
    "updated_date", "created_date", "indexed_in", "apc_list", "apc_paid",
    "has_fulltext", "has_content", "content_urls", "display_name"
}


def extract_authors(authorships):
    if not isinstance(authorships, list):
        return []
    authors = []
    for a in authorships:
        author_info = a.get("author", {})
        name = author_info.get("display_name")
        if name:
            authors.append(name)
    return authors


def extract_journal(primary_location):
    if not isinstance(primary_location, dict):
        return ""
    source = primary_location.get("source", {})
    if isinstance(source, dict):
        return source.get("display_name", "")
    return ""


def extract_concepts(concepts):
    if not isinstance(concepts, list):
        return []
    return [c.get("display_name") for c in concepts if isinstance(c, dict) and c.get("display_name")]


def clean_paper(paper):
    cleaned = {}

    # Core Screening & Metadata
    cleaned["id"] = paper.get("id", "")
    cleaned["doi"] = paper.get("doi", "")
    cleaned["title"] = paper.get("title", "")
    cleaned["abstract"] = paper.get("abstract", "")
    cleaned["publication_year"] = paper.get("publication_year")
    cleaned["publication_date"] = paper.get("publication_date", "")
    cleaned["type"] = paper.get("type", "")
    cleaned["language"] = paper.get("language", "")

    # Custom Pipeline Fields
    cleaned["_method"] = paper.get("_method", "")
    cleaned["_abm_score"] = paper.get("_abm_score")

    # Extracted Flattened Info
    cleaned["authors"] = extract_authors(paper.get("authorships"))
    cleaned["journal"] = extract_journal(paper.get("primary_location"))
    cleaned["concepts"] = extract_concepts(paper.get("concepts"))

    # Impact & Citations
    cleaned["cited_by_count"] = paper.get("cited_by_count", 0)
    cleaned["fwci"] = paper.get("fwci")

    # Retain remaining valid fields
    for k, v in paper.items():
        if k not in KEYS_TO_DROP and k not in cleaned:
            cleaned[k] = v

    return cleaned


# ==========================
# 2. In-Memory Process & Save
# ==========================
cleaned_papers = [clean_paper(p) for p in tqdm(papers, desc="In-Memory Cleaning")]

# Save cleaned dataset
CLEANED_OUTPUT_PATH = r"P:\Literature-Research-Agent-main\repository\cleaned_literature.json"
with open(CLEANED_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(cleaned_papers, f, ensure_ascii=False, indent=2)

print(f"Dataset successfully cleaned and saved! Total records: {len(cleaned_papers)}")

In-Memory Cleaning: 100%|██████████| 64500/64500 [00:03<00:00, 20546.93it/s]


Dataset successfully cleaned and saved! Total records: 64500
